# 基于MindSpore NLP的T5文本摘要微调与推理

本 Notebook 演示如何在 **MindSpore 2.7.0** 与 **MindSpore NLP 0.5.1** 环境下，对 `t5-base` 进行摘要任务微调，并完成推理与保存。

- **任务**：基于 `abisee/cnn_dailymail`（`3.0.0`）数据集进行文本摘要微调（可迁移到“日常邮件摘要”场景）。
- **目标**：在不改动训练逻辑的前提下，跑通「数据 → 分词 → 数据管线 → 训练 → 推理 → 保存」全流程。
- **输出**：将微调后的模型与分词器保存至 `./t5_email_summarization_ms270_mn051` 目录，便于后续加载推理或继续训练。

本示例以快速验证为目标：训练仅运行 **1 个 epoch** 且最多 **200 step**，并每 20 step 打印一次 loss。


## 环境准备

安装本示例依赖的 MindSpore NLP 版本。请确保当前 Python 环境中已安装 **MindSpore 2.7.0**（或与之兼容的版本）。


In [ ]:
!pip install mindnlp==0.5.1

## 版本检查

打印当前运行环境中的 MindSpore 与 MindSpore NLP 版本号，便于定位依赖不一致导致的兼容性问题。


In [ ]:
import mindspore as ms
import mindnlp
print('MindSpore:', ms.__version__)
print('MindNLP:', getattr(mindnlp, '__version__', 'unknown'))


## 版本显示差异说明（MindSpore NLP 0.5.1 安装后显示为 0.5.0rc2）

![alt text](image.png)

在部分环境中，`pip install mindnlp==0.5.1` 后 `mindnlp.__version__` 可能显示为 `0.5.0rc2`。若出现该情况，建议优先以 `pip show mindnlp` 与已安装 wheel 信息为准，并结合 `import mindnlp; mindnlp.__file__` 核对实际导入路径，确认是否存在旧版本残留或多环境混用。


## 数据集加载

使用 `mindnlp.dataset.load_dataset` 加载经典摘要数据集 **CNN/DailyMail**（版本 `3.0.0`），并获取 `train/validation/test` 三个子集。运行后打印各子集样本数量，用于确认下载与加载是否正常。


In [ ]:
from mindnlp.dataset import load_dataset
ds = load_dataset('abisee/cnn_dailymail', '3.0.0', split=['train', 'validation', 'test'])
train = ds['train']
val = ds['validation']
test = ds['test']
print('train:', len(train), 'validation:', len(val), 'test:', len(test))

## 分词器与编码设置

加载 `t5-base` 的预训练分词器，并设置摘要任务常用的编码参数：

- `prefix = "summarize: "`：T5 摘要任务前缀（prompt）。
- `max_input_length`：输入文本最大长度（示例为 512）。
- `max_target_length`：目标摘要最大长度（示例为 64）。
- `pad_token_id`：用于 padding 的 token ID（后续需要在 label 中将 pad 位置替换为 `-100` 以忽略损失）。


In [ ]:
from mindnlp.transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('t5-base')
prefix = 'summarize: '
max_input_length = 512
max_target_length = 64
pad_id = tokenizer.pad_token_id
print('pad_token_id:', pad_id)

## 生成式数据集

定义自定义数据集类 `SummarizationDataset`，将原始样本转为模型可训练的三元组：

- `input_ids`：编码后的输入 token 序列
- `attention_mask`：注意力 mask
- `labels`：编码后的摘要 token 序列（将 `pad_token_id` 替换为 `-100`，使损失函数忽略 padding 部分）

该步骤的核心是将“文本样本”转换为“可直接送入 T5 的张量输入”。


In [ ]:
import numpy as np
class SummarizationDataset:
    def __init__(self, data):
        self.data = data
    def __getitem__(self, idx):
        art, summ, _ = self.data[idx]
        enc = tokenizer(prefix + art, max_length=max_input_length, padding='max_length', truncation=True)
        dec = tokenizer(summ, max_length=max_target_length, padding='max_length', truncation=True)
        input_ids = np.array(enc.input_ids, dtype=np.int32)
        attention_mask = np.array(enc.attention_mask, dtype=np.int32)
        labels = np.array(dec.input_ids, dtype=np.int32)
        labels = np.where(labels == pad_id, -100, labels).astype(np.int32)
        return input_ids, attention_mask, labels
    def __len__(self):
        return len(self.data)

## 构建数据处理流水线

将 `SummarizationDataset` 封装为 MindSpore `GeneratorDataset`，并完成：

- 类型转换：统一 cast 为 `int32`
- 批处理：设置 `batch_size=8`
- 形状对齐：`drop_remainder=True`，保证 batch 维度固定，便于训练稳定运行


In [ ]:
import mindspore.dataset as ds
from mindspore.dataset import transforms
import mindspore
train_ds = ds.GeneratorDataset(SummarizationDataset(train), column_names=['input_ids','attention_mask','labels'], shuffle=True)
val_ds = ds.GeneratorDataset(SummarizationDataset(val), column_names=['input_ids','attention_mask','labels'], shuffle=False)
for c in ['input_ids','attention_mask','labels']:
    train_ds = train_ds.map(operations=transforms.TypeCast(mindspore.int32), input_columns=c)
    val_ds = val_ds.map(operations=transforms.TypeCast(mindspore.int32), input_columns=c)
batch_size = 8
train_ds = train_ds.batch(batch_size, drop_remainder=True)
val_ds = val_ds.batch(batch_size, drop_remainder=True)
print('train batches:', train_ds.get_dataset_size(), 'val batches:', val_ds.get_dataset_size())

## 构建模型与训练函数

本节完成微调核心逻辑的搭建，包括：

- 加载 `T5ForConditionalGeneration`（`t5-base`）
- 构建优化器与梯度裁剪策略
- 处理 `mindtorch.autograd.profiler` 的兼容性引用（用于避免部分环境缺失导致的报错）
- 定义 `train_step`：封装单步训练流程（前向 → loss → 反向 → 裁剪 → 更新）

说明：该 Notebook 保持“代码逻辑不变”，仅整理文字结构以提升可读性。


In [ ]:
import contextlib
import mindtorch.autograd as autograd
import mindtorch as mt
from mindnlp.transformers import T5ForConditionalGeneration

try:
    import mindtorch.autograd.profiler as _prof
    autograd.profiler = _prof
except Exception:
    class _DummyProfiler:
        @staticmethod
        @contextlib.contextmanager
        def record_function(name):
            yield
    autograd.profiler = _DummyProfiler()


model = T5ForConditionalGeneration.from_pretrained("t5-base")

model.train()


lr = 1e-4
weight_decay = 0.01
optimizer = mt.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)


clip_norm = 1.0

def to_long(x):
    if hasattr(x, "asnumpy"):
        x = x.asnumpy()
    return mt.tensor(x, dtype=mt.long)

def train_step(input_ids, attention_mask, labels):
    input_ids = to_long(input_ids)
    attention_mask = to_long(attention_mask)
    labels = to_long(labels)

    optimizer.zero_grad()
    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    loss = outputs.loss

    loss.backward()
    mt.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
    optimizer.step()

    return float(loss.item())


## 训练

启动训练循环：

- `num_epochs = 1`：仅训练 1 个 epoch，用于快速验证流程
- `warm_steps = 200`：每个 epoch 最多跑 200 个 step
- 日志：每 20 step 打印一次 loss，便于观察收敛趋势

如果需要正式训练，可按实际数据规模与资源条件调整 epoch、step、batch size 与学习率等超参数。


In [ ]:
from tqdm import tqdm

num_epochs = 1
warm_steps = 200

for epoch in range(num_epochs):
    step = 0
    with tqdm(total=warm_steps, desc=f"Epoch {epoch+1}/{num_epochs}", unit="step") as pbar:
        for input_ids, attention_mask, labels in train_ds.create_tuple_iterator():
            loss = train_step(input_ids, attention_mask, labels)  
            step += 1

            if step % 20 == 0:
                tqdm.write(f"step={step}, loss={loss:.4f}")

            pbar.update(1)
            if step >= warm_steps:
                break


## 推理

构造一段较长输入文本，按 T5 摘要任务要求加上前缀 `summarize: `，并调用 `model.generate` 生成摘要序列。最后通过分词器 `decode` 将生成的 token ID 还原为可读文本，用于快速验证微调后的生成效果。


In [ ]:
text = """Artificial Intelligence (AI) has rapidly evolved and become an integral part of various industries, 
particularly healthcare. AI's ability to analyze large volumes of data and identify patterns that humans might miss 
has revolutionized many aspects of medical care, 
from diagnostics to treatment planning and patient monitoring.
One of the most significant contributions of AI in healthcare is in the field of diagnostics. 
Machine learning algorithms, a subset of AI, have been trained to interpret medical imaging data, such as X-rays, MRIs, and CT scans.
These algorithms can often detect abnormalities, such as tumors or fractures, with higher accuracy and speed than human radiologists. 
This not only improves the speed of diagnosis but also reduces the potential for human error, leading to better patient outcomes.
AI has also shown promise in personalized medicine, where treatments are tailored to an individual's genetic makeup and health history.
By analyzing patient data, including genetic information, 
AI systems can recommend specific treatment plans that are more likely to be effective. 
This can greatly enhance the precision of treatments, reduce side effects, and improve overall patient care.."""
enc = tokenizer(prefix + text, max_length=max_input_length, padding='max_length', truncation=True)
inp = ms.Tensor([enc.input_ids], dtype=ms.int32)
gen_ids = model.generate(inp, max_length=60,top_k=0,temperature=0.7)
summ = tokenizer.decode(gen_ids[0], skip_special_tokens=True)
print(summ)

## 保存模型与分词器

将微调后的模型权重与分词器配置保存到本地目录：

- 保存路径：`./t5_email_summarization_ms270_mn051`
- 保存内容：模型参数（权重）与 tokenizer 配置/词表

保存后可在后续脚本中通过 `from_pretrained(save_dir)` 直接加载，便于部署推理或继续训练。


In [ ]:
save_dir = './t5_email_summarization_ms270_mn051'
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print('saved to', save_dir)